# Step 2 — Fine-tune PANNs on ESC-50

Reproduces **Table II** and **Table III** from the paper.

Trains a logistic regression head on PANNs CNN14 embeddings.

**Output:** `models/esas_finetuned.pkl`

**Time:** ~15 minutes on CPU (cached after first run).

## Configuration

**Change `BASE` below to match your machine before running anything else.**
All other paths are derived automatically.

In [ ]:
from pathlib import Path

# ── Set your base path here ───────────────────────────────────────────────
# Change this to where your esas_project folder is on your machine.
# Everything else is derived automatically.
BASE = Path('/path/to/your/esas_project')  # <-- set this
# ─────────────────────────────────────────────────────────────────────────

RECORDINGS = BASE / 'recordings'
ESC50_DIR  = BASE / 'ESC-50'
PANNS_CKPT = BASE / 'panns_data' / 'Cnn14_mAP=0.431.pth'
MODELS_DIR = Path('models')  # saved inside esas_clean
RESULTS_DIR = Path('results')  # saved inside esas_clean
MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
print(f'BASE:       {BASE}')
print(f'Recordings: {RECORDINGS.exists()}')
print(f'ESC-50:     {ESC50_DIR.exists()}')
print(f'PANNs:      {PANNS_CKPT.exists()}')

In [ ]:
import csv, glob, json, pickle, warnings
from collections import Counter
import numpy as np
import librosa
from panns_inference import AudioTagging
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import precision_recall_fscore_support
warnings.filterwarnings('ignore')
print('Imports OK')

## Hazard taxonomy (Table I)

In [ ]:
HIGH   = {'car_horn','chainsaw','crackling_fire','glass_breaking','hand_saw','siren','fireworks'}
MEDIUM = {'clock_alarm','clock_tick','crying_baby','dog','door_wood_knock','sneezing',
           'coughing','church_bells','vacuum_cleaner','washing_machine','toilet_flush','cat'}

def get_priority(label):
    if label in HIGH:   return 'HIGH'
    if label in MEDIUM: return 'MEDIUM'
    return 'LOW'

meta = {}
with open(ESC50_DIR / 'meta' / 'esc50.csv') as f:
    for row in csv.DictReader(f):
        meta[row['filename']] = {'cat': row['category'], 'fold': int(row['fold'])}

dist = Counter(get_priority(v['cat']) for v in meta.values())
print(f'ESC-50: {len(meta)} clips  HIGH={dist["HIGH"]}  MEDIUM={dist["MEDIUM"]}  LOW={dist["LOW"]}')

## Extract PANNs embeddings (cached after first run)

In [ ]:
CACHE = MODELS_DIR / 'embeddings_cache.npz'

if CACHE.exists():
    print(f'Loading cached embeddings from {CACHE}')
    d = np.load(CACHE, allow_pickle=True)
    X, y, folds = d['X'], d['y'], d['folds']
else:
    assert PANNS_CKPT.exists(), f'Run notebook 01 first'
    panns = AudioTagging(checkpoint_path=str(PANNS_CKPT), device='cpu')
    files = sorted(glob.glob(str(ESC50_DIR / 'audio' / '*.wav')))
    print(f'Extracting from {len(files)} clips (~15 min)...')
    X_list, y_list, folds_list = [], [], []
    for i, path in enumerate(files):
        fn = Path(path).name
        info = meta.get(fn)
        if info is None: continue
        try:
            audio, _ = librosa.load(path, sr=32000, mono=True)
            audio = audio[:64000] if len(audio)>=64000 else np.pad(audio,(0,64000-len(audio)))
            clipwise, embedding = panns.inference(audio[None,:])
            # embedding shape is (1, 2048) — take first row
            emb = embedding[0] if embedding.ndim == 2 else embedding
            X_list.append(emb)
            y_list.append(get_priority(info['cat']))
            folds_list.append(info['fold'])
        except Exception as e:
            print(f'  Skip {fn}: {e}')
        if (i+1)%400==0: print(f'  {i+1}/{len(files)}  embeddings so far: {len(X_list)}')
    X     = np.array(X_list)
    y     = np.array(y_list)
    folds = np.array(folds_list)
    print(f'Extracted {len(X)} embeddings')
    np.savez(CACHE, X=X, y=y, folds=folds)
    print(f'Cached to {CACHE}')

from collections import Counter
print(f'Embeddings: {X.shape}  {Counter(y)}')


## 5-fold cross-validation — Table II

In [ ]:
le = LabelEncoder()
ye = le.fit_transform(y)
all_true, all_pred = [], []

for fold in range(1, 6):
    tr = folds != fold
    te = folds == fold
    clf = LogisticRegression(C=1.0, max_iter=1000, random_state=42, class_weight='balanced')
    clf.fit(X[tr], ye[tr])
    all_pred.extend(clf.predict(X[te]))
    all_true.extend(ye[te])
    print(f'Fold {fold}/5 done')

all_true = np.array(all_true)
all_pred = np.array(all_pred)
classes  = le.classes_

In [ ]:
# Print per-fold F1 to use in statistics notebook
from sklearn.metrics import f1_score
print("Per-fold macro F1 scores:")
all_true2, all_pred2 = [], []
for fold in range(1, 6):
    tr = folds != fold
    te = folds == fold
    clf2 = LogisticRegression(C=1.0, max_iter=1000, random_state=42, class_weight='balanced')
    clf2.fit(X[tr], ye[tr])
    preds = clf2.predict(X[te])
    fold_f1 = f1_score(ye[te], preds, average='macro')
    print(f"  Fold {fold}: {fold_f1:.4f}")

## Results — Table II

In [ ]:
p, r, f1_scores, _ = precision_recall_fscore_support(all_true, all_pred,
               labels=range(len(classes)), zero_division=0)
mp, mr, mf1, _ = precision_recall_fscore_support(all_true, all_pred,
                 average='macro', zero_division=0)

print('='*52)
print('  TABLE II — Detection Results (5-fold CV)')
print('='*52)
print(f'  {"Priority":<10} {"P":>8} {"R":>8} {"F1":>8}')
print(f'  {"-"*36}')
for i, cls in enumerate(classes):
    print(f'  {cls:<10} {p[i]:>8.3f} {r[i]:>8.3f} {f1_scores[i]:>8.3f}')
print(f'  {"-"*36}')
print(f'  {"Macro":<10} {mp:>8.3f} {mr:>8.3f} {mf1:>8.3f}')
print('='*52)

## Train on full dataset and save model

In [ ]:
clf_final = LogisticRegression(C=1.0, max_iter=1000, random_state=42, class_weight='balanced')
clf_final.fit(X, ye)
clf_final.classes_ = classes

with open(MODELS_DIR / 'esas_finetuned.pkl', 'wb') as pkl_file:
    pickle.dump(clf_final, pkl_file)
print(f'Saved: {MODELS_DIR}/esas_finetuned.pkl')

In [ ]:
import json

p2, r2, f2, _ = precision_recall_fscore_support(
    all_true, all_pred, labels=range(len(classes)), zero_division=0)
mp2, mr2, mf2, _ = precision_recall_fscore_support(
    all_true, all_pred, average='macro', zero_division=0)

results = {}
for i, cls in enumerate(classes):
    results[cls] = {'P': round(float(p2[i]),3),
                    'R': round(float(r2[i]),3),
                    'F1': round(float(f2[i]),3)}
results['Macro'] = {'P': round(float(mp2),3),
                    'R': round(float(mr2),3),
                    'F1': round(float(mf2),3)}

with open(RESULTS_DIR / 'esc50_results.json', 'w') as json_file:
    json.dump(results, json_file, indent=2)
print(f'Saved: {RESULTS_DIR}/esc50_results.json')
print(results)